#  ISL Dataset Preparation
**Goal:** Prepare Indian Sign Language (ISL) dataset for training a gesture recognition model.

# Dataset Options
1. Use an existing ISL image dataset (Kaggle)- Dataset Link: https://www.kaggle.com/datasets/soumyakushwaha/indian-sign-language-dataset?select=ISL_Dataset
2. Collect your own ISL samples using webcam
3. Combine both approaches (best results)

Import Libraries

In [14]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil

##  Using Local ISL Dataset
We are using the Kaggle ISL dataset **already downloaded locally**.

In [21]:
DATASET_PATH = Path("/Users/vidhikadam/Downloads/ISL_Dataset")

print(" Dataset path set to:")
print(DATASET_PATH)

 Dataset path set to:
/Users/vidhikadam/Downloads/ISL_Dataset


## Verify Dataset Structure

In [60]:
print("📁 Dataset structure:\n")

for item in DATASET_PATH.iterdir():
    if item.is_dir():
        print("📂", item.name)

📁 Dataset structure:

📂 R
📂 U
📂 I
📂 N
📂 G
📂 Z
📂 T
📂 S
📂 A
📂 F
📂 O
📂 M
📂 C
📂 D
📂 V
📂 Q
📂 X
📂 E
📂 B
📂 K
📂 L
📂 P
📂 W


## Method 1: Convert ISL Images to Hand Landmarks

We use **MediaPipe Hands** to extract hand landmarks and convert images into
numerical feature vectors compatible with our existing gesture model.

In [63]:
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

sys.path.append('../src')
from feature_extractor import HandFeatureExtractor

feature_extractor = HandFeatureExtractor()
print("MediaPipe & Feature Extractor ready")

MediaPipe & Feature Extractor ready


In [65]:
def extract_landmarks_from_image(image_path):

    image = cv2.imread(str(image_path))
    if image is None:
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            return feature_extractor.extract_features(hand_landmarks)

    return None

## Create ISL Dataset from Images

This converts **A–Z image folders** into:
- Feature matrix `X`
- Label vector `y`
- Class names

In [70]:
def create_isl_dataset_from_images(
    dataset_path,
    output_path='../data/raw/isl_dataset.npz'
):
    print("Processing ISL images...\n")

    features_list = []
    labels_list = []
    label_names = []

    dataset_root = Path(dataset_path)

    class_folders = sorted([f for f in dataset_root.iterdir() if f.is_dir()])

    print(f"Found {len(class_folders)} classes")

    for class_idx, class_folder in enumerate(class_folders):
        label_names.append(class_folder.name)
        print(f"\n📂 Class {class_idx}: {class_folder.name}")

        image_files = list(class_folder.glob("*.jpg")) + \
                      list(class_folder.glob("*.png")) + \
                      list(class_folder.glob("*.jpeg"))

        success = 0

        for img in image_files:
            features = extract_landmarks_from_image(img)
            if features is not None:
                features_list.append(features)
                labels_list.append(class_idx)
                success += 1

        print(f"   {success}/{len(image_files)} images processed")

    X = np.array(features_list)
    y = np.array(labels_list)

    print("\nFinal Dataset Summary")
    print("Features shape:", X.shape)
    print("Labels shape:", y.shape)

    os.makedirs(Path(output_path).parent, exist_ok=True)
    np.savez(output_path, features=X, labels=y, class_names=label_names)

    print(f"\n💾 Dataset saved at: {output_path}")
    return X, y, label_names

##  Run Dataset Processing

In [73]:
X, y, class_names = create_isl_dataset_from_images(
    dataset_path=DATASET_PATH
)

Processing ISL images...

Found 23 classes

📂 Class 0: A


I0000 00:00:1766817936.322875       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1766817936.394964       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.436935       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.477810       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.515152       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.554967       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.595678       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.631949       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817936.656657       1 gl_context

   19/25 images processed

📂 Class 1: B


I0000 00:00:1766817937.439207       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.479650       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.517945       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.541819       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.577825       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.613696       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.650178       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.690297       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817937.726719       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: A

   23/38 images processed

📂 Class 2: C


...

I0000 00:00:1766817959.647180       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.688430       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.729090       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.766894       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.802822       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.845636       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.883423       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.920884       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817959.963033       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: A

   16/24 images processed

📂 Class 22: Z


I0000 00:00:1766817960.534422       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.559680       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.595883       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.631927       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.669304       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.715744       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.748926       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.796059       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817960.833398       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: A

   17/27 images processed

Final Dataset Summary
Features shape: (507, 57)
Labels shape: (507,)

💾 Dataset saved at: ../data/raw/isl_dataset.npz


I0000 00:00:1766817961.421863       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
I0000 00:00:1766817961.450229       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2


##  Method 2: Collect Your Own ISL Data (Optional)

This allows you to:
- Create a **custom dataset**

In [77]:
# ISL Alphabet
ISL_ALPHABET = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

print("ISL Alphabets:", len(ISL_ALPHABET))

ISL Alphabets: 26
